# Finaliser les mesures du POC

Je mesure le socle et le SFT v3 sur les partitions de test figées, puis je vérifie l’API et vLLM sur trois cas fictifs. Le DPO est également mesuré si ses poids originaux sont présents sur Drive et correspondent à leur empreinte connue. Aucun entraînement n’est relancé.

```text
protocole figé → test QA / triage → vLLM + API → traces et MLflow → sauvegarde → libération GPU
```

La cellule nécessite une G4 connectée et le montage de Drive. Je conserve toutes les sorties dans `evaluations/finalisation_20261105`, ainsi qu’une archive `ocr14_resultats_finaux_20261105.zip`. Les échecs sont enregistrés ; ils ne valent pas validation. Le test vLLM est temporaire et interne à la VM : il ne crée pas un endpoint public permanent.

Je ne change pas le candidat retenu après lecture du test. Je n’interprète pas les trois cas fictifs comme une mesure de pertinence clinique. [Installation isolée recommandée par vLLM](https://docs.vllm.ai/en/v0.32.0/getting_started/installation/gpu/).


In [ ]:
# Je termine les mesures figées, sauvegarde les preuves, puis libère la VM.
from google.colab import drive, runtime
from pathlib import Path
import hashlib
import json
import os
import shutil
import sqlite3
import subprocess
import sys
import traceback
import zipfile

drive.mount('/content/drive')
DRIVE = Path('/content/drive/MyDrive/OCR_PROJET_14')
PAQUET = DRIVE / 'ocr14_finalisation_20261105.zip'
RACINE = Path('/content/OCR14_finalisation')
SORTIES = DRIVE / 'evaluations/finalisation_20261105'
SORTIES.mkdir(parents=True, exist_ok=True)
BASE = Path('/content/ocr14_finalisation_mlflow.db')
BASE_SOURCE = DRIVE / 'suivi/mlflow_colab.db'
BASE_SORTIE = SORTIES / 'mlflow_finalisation.db'
STATUT = {'test_final': 'non_execute', 'vllm': 'non_execute', 'dpo_poids': 'a_verifier'}
processus_installation = None

def executer(commande, journal):
    """Conserve le journal et affiche les dernières lignes uniquement en cas d'échec."""
    print('Étape :', journal.name, flush=True)
    with journal.open('a', encoding='utf-8') as log:
        retour = subprocess.run(commande, cwd=RACINE, stdout=log, stderr=subprocess.STDOUT,
                                env={**os.environ, 'PYTHONUNBUFFERED': '1'}).returncode
    if retour:
        print(journal.read_text(errors='replace')[-7000:])
        raise RuntimeError(f'Code {retour} ; journal : {journal.name}')

try:
    assert hashlib.sha256(PAQUET.read_bytes()).hexdigest() == '85cd3909bb3edf4f0ae73eb4ce85de74a08dd14587fbd0622a36da9b67372a95', 'Paquet différent.'
    RACINE.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(PAQUET) as archive:
        assert all((RACINE / nom).resolve().is_relative_to(RACINE.resolve()) for nom in archive.namelist())
        archive.extractall(RACINE)
    executer([sys.executable, '-m', 'pip', 'install', '-q', 'uv',
        'transformers==5.22.1', 'peft==0.21.2', 'accelerate==1.15.0',
        'scikit-learn==1.9.1', 'mlflow==3.17.2'], SORTIES / 'installation_evaluation.log')
    # torchao ne sert pas à cette inférence BF16 et a bloqué PEFT lors d'une session précédente.
    executer([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], SORTIES / 'installation_evaluation.log')
    gpu = subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], text=True)
    print('GPU attribué :', gpu.strip(), flush=True)
    (SORTIES / 'gpu.txt').write_text(gpu)
    executer([sys.executable, '-c', 'import torch; assert torch.cuda.is_available() and torch.cuda.is_bf16_supported()'],
             SORTIES / 'verification_gpu.log')
    if not BASE.exists() and BASE_SOURCE.exists():
        shutil.copy2(BASE_SOURCE, BASE)
    SFT = DRIVE / 'experiences/sft_dpo_v3_20261105/sft_triage/adapter'
    DPO = DRIVE / 'experiences/sft_dpo_v3_20261105/dpo_pilote/adapter'
    assert (SFT / 'adapter_model.safetensors').exists(), 'Poids SFT absents de Drive.'
    ADAPTATEUR = RACINE / 'models/sft_triage_v3/adapter'
    shutil.copytree(SFT, ADAPTATEUR, dirs_exist_ok=True)
    commande = [sys.executable, '-m', 'evaluation.test_final', '--data-dir', str(RACINE / 'data/processed/partitions_v3'),
        '--adapter', str(ADAPTATEUR), '--output', str(SORTIES / 'test'),
        '--tracking-uri', f'sqlite:///{BASE}', '--artifact-location', (SORTIES / 'artefacts_mlflow').as_uri()]
    if (DPO / 'adapter_model.safetensors').exists():
        sha_dpo = hashlib.sha256((DPO / 'adapter_model.safetensors').read_bytes()).hexdigest()
        assert sha_dpo == 'ae3cb4d55f3df9df573b031ccc8842fae9bbbddf2c634bd2e95fe2486d51cecc', 'DPO différent de celui évalué.'
        DPO_LOCAL = RACINE / 'models/dpo_pilote_v3/adapter'
        shutil.copytree(DPO, DPO_LOCAL, dirs_exist_ok=True)
        commande += ['--dpo-adapter', str(DPO_LOCAL)]
        shutil.copytree(DPO_LOCAL, SORTIES / 'poids_dpo_recuperes', dirs_exist_ok=True)
        STATUT['dpo_poids'] = 'recuperes_et_identiques'
    else:
        STATUT['dpo_poids'] = 'absents_sur_Drive_test_limite_au_socle_et_SFT'
    # J'isole vLLM de l'environnement Transformers utilisé pour les mesures.
    executer([sys.executable, '-m', 'uv', 'venv', '--python', '3.12', '--seed', '--managed-python',
        '/content/ocr14_vllm_env'], SORTIES / 'installation_vllm.log')
    PY_VLLM = '/content/ocr14_vllm_env/bin/python'
    with (SORTIES / 'installation_vllm.log').open('a') as log:
        processus_installation = subprocess.Popen([sys.executable, '-m', 'uv', 'pip', 'install',
            '--python', PY_VLLM, 'vllm==0.32.0', 'uvicorn==0.54.0', 'jinja2==3.1.6', '--torch-backend=auto'],
            stdout=log, stderr=subprocess.STDOUT)
    executer(commande, SORTIES / 'test_final.log')
    STATUT['test_final'] = 'termine'
    print('Les évaluations finales sont terminées.', flush=True)
    assert processus_installation.wait() == 0, 'Installation vLLM échouée : voir installation_vllm.log.'
    executer(['/content/ocr14_vllm_env/bin/vllm', 'download-kernels'], SORTIES / 'kernels_vllm.log')
    executer([PY_VLLM, '-m', 'serving.verifier_colab', '--adapter', str(ADAPTATEUR),
        '--output', str(SORTIES / 'vllm')], SORTIES / 'verification_vllm.log')
    STATUT['vllm'] = 'verifie_sur_trois_cas_fictifs'
except Exception:
    STATUT['erreur'] = traceback.format_exc()
    print(STATUT['erreur'], flush=True)
finally:
    if processus_installation is not None and processus_installation.poll() is None:
        processus_installation.terminate()
        processus_installation.wait(timeout=30)
    if BASE.exists():
        with sqlite3.connect(BASE) as source, sqlite3.connect(BASE_SORTIE) as cible:
            source.backup(cible)
    (SORTIES / 'statut.json').write_text(json.dumps(STATUT, ensure_ascii=False, indent=2) + '\n')
    # Une archive unique facilite la récupération intégrale, y compris des preuves d'échec.
    archive_locale = Path('/content/ocr14_resultats_finaux_20261105.zip')
    with zipfile.ZipFile(archive_locale, 'w', zipfile.ZIP_DEFLATED) as archive:
        for fichier in sorted(SORTIES.rglob('*')):
            if fichier.is_file():
                archive.write(fichier, fichier.relative_to(SORTIES))
    archive_drive = DRIVE / archive_locale.name
    shutil.copy2(archive_locale, archive_drive)
    assert hashlib.sha256(archive_drive.read_bytes()).digest() == hashlib.sha256(archive_locale.read_bytes()).digest()
    print(json.dumps(STATUT, ensure_ascii=False, indent=2), flush=True)
    print('Résultats et suivi sauvegardés. Synchronisation Drive puis libération du GPU.', flush=True)
    drive.flush_and_unmount(timeout_ms=300_000)
    runtime.unassign()
